In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision transformers matplotlib pillow -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["torch", "torchvision", "transformers", "matplotlib", "pillow"]:
    _install(pkg)

In [ ]:
import hashlib
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# LSeg 实战：架构解析 + 推理应用

基于论文 *Language-driven Semantic Segmentation*，本 Notebook 用 **两条互补路径** 理解 LSeg：

1. 用一个 **toy LSeg** 拆解“冻结文本特征 + 像素级相似度 + 有监督分割损失”的训练逻辑。
2. 用 **CLIPSeg 预训练模型** 演示真正可运行的 zero-shot / open-vocabulary 分割推理。

| | 架构解析 | 推理应用 |
|---|---|---|
| 核心思路 | 逐组件拆解 LSeg 的关键模块与张量流 | 加载 `CIDAS/clipseg-rd64-refined` 直接做文本驱动分割 |
| 重点 | 为什么要把文本特征做到像素级 | 如何把 `prompt -> mask` 这条链路跑通 |
| 适合场景 | 面试准备、原理理解 | 快速实验、工程原型 |
| 实现策略 | toy supervision demo | pretrained inference demo |

## Notebook 导航与实现策略

- 理论输入：`04_LSeg.md`
- 实现策略：**C — 架构解析 + 预训练推理应用**
- 选择理由：LSeg 的完整训练依赖像素级分割标注与专门训练流程，不适合在 CPU-only Notebook 中直接复现；因此这里用 **toy LSeg** 解释训练/推理差异，再用 **CLIPSeg** 演示可直接运行的开放词汇分割。

### 已核对的 API 与事实

- `CLIPSegProcessor` / `CLIPSegForImageSegmentation` 支持：
  `processor(text=texts, images=[image] * len(texts), padding=True, return_tensors="pt")`，输出 `outputs.logits`，形状为 `(num_prompts, 352, 352)`。
- `nn.TransformerEncoderLayer(..., batch_first=True)` 可以直接处理 `(batch, seq, d_model)` 形状。
- LSeg 的核心思路是：冻结文本侧语义表征，让图像侧产生密集像素特征，再通过像素-文本相似度做分割；训练阶段使用有监督分割目标，推理阶段允许输入任意 prompt。

### 本 Notebook 结构

1. 数据准备：toy 合成分割样本 + 真实图片
2. 共享组件：超参数、可视化与文本特征工具
3. 架构解析：文本编码器、DPT 风格图像编码器、像素级相似度、SRB、toy LSeg
4. 推理应用：CLIPSeg zero-shot segmentation
5. 面试拓展：高频问答与模型对比

> 注：toy LSeg 只用于解释代码路径，不代表真实的零样本泛化能力；真正的开放词汇能力来自预训练的视觉-语言模型。

## 1. 数据准备

本节准备两类输入：

1. **toy segmentation batch**：用于演示 LSeg 的训练路径，包含像素级标签。
2. **real image + text prompts**：用于演示预训练模型的实际 zero-shot 推理。

这样可以把“训练机制”和“真实应用”拆开看：
- toy 数据负责解释 supervision 从哪里来；
- 真实图片负责展示 prompt 如何直接驱动 mask 生成。

In [ ]:
TOY_CLASS_NAMES = ["background", "red region", "green region", "blue region"]
REAL_PROMPTS = ["a cat", "a remote control", "a blanket", "a couch"]
REAL_IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"


def make_toy_batch(batch_size=8, image_size=64):
    """生成一个 CPU 友好的 toy segmentation batch。"""
    images = torch.zeros(batch_size, 3, image_size, image_size)
    masks = torch.zeros(batch_size, image_size, image_size, dtype=torch.long)

    yy, xx = torch.meshgrid(
        torch.arange(image_size),
        torch.arange(image_size),
        indexing="ij",
    )

    for idx in range(batch_size):
        images[idx] += 0.05 * torch.rand(3, image_size, image_size)

        # class 1: red rectangle
        x1 = np.random.randint(4, image_size // 2)
        y1 = np.random.randint(4, image_size // 2)
        w1 = np.random.randint(image_size // 6, image_size // 3)
        h1 = np.random.randint(image_size // 6, image_size // 3)
        x2 = min(image_size, x1 + w1)
        y2 = min(image_size, y1 + h1)
        masks[idx, y1:y2, x1:x2] = 1
        images[idx, 0, y1:y2, x1:x2] = 1.0

        # class 2: green circle
        cx = np.random.randint(image_size // 3, 2 * image_size // 3)
        cy = np.random.randint(image_size // 3, 2 * image_size // 3)
        r = np.random.randint(image_size // 8, image_size // 5)
        circle = (xx - cx).pow(2) + (yy - cy).pow(2) <= r ** 2
        masks[idx][circle] = 2
        images[idx, 1][circle] = 1.0

        # class 3: blue diagonal band
        offset = np.random.randint(-8, 8)
        band_width = np.random.randint(3, 6)
        band = (yy - xx - offset).abs() < band_width
        masks[idx][band] = 3
        images[idx, 2][band] = 1.0

    return images.clamp(0, 1), masks

### 1.1 生成 toy segmentation 样本

这里不下载真实分割数据集，而是直接生成一个 **小型彩色形状分割任务**：
- 输入：`(B, 3, H, W)` 彩色图像
- 标签：`(B, H, W)` 像素类别图

它的作用不是追求高精度，而是让我们清楚看到：
**LSeg 训练时必须有像素标签，损失函数才能落到每个像素上。**

In [ ]:
toy_images, toy_masks = make_toy_batch(batch_size=6, image_size=64)

print(f"toy_images shape: {tuple(toy_images.shape)}")   # (B, 3, H, W)
print(f"toy_masks shape:  {tuple(toy_masks.shape)}")    # (B, H, W)
print(f"class ids: {sorted(torch.unique(toy_masks).tolist())}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for idx, ax in enumerate(axes.flat):
    ax.imshow(np.transpose(toy_images[idx].numpy(), (1, 2, 0)))
    ax.imshow(toy_masks[idx].numpy(), alpha=0.28, cmap="viridis", vmin=0, vmax=3)
    ax.set_title(f"Toy sample {idx}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 1.2 准备真实图片用于 zero-shot 推理

为了演示真正的开放词汇分割，我们加载一张公开图片。后面会把同一张图与多个文本 prompt 配对输入 CLIPSeg。

这一步对应实际应用里的 `raw image -> processor -> model` 起点。

In [ ]:
with urlopen(REAL_IMAGE_URL) as response:
    real_image = Image.open(response).convert("RGB")

print(f"Real image size: {real_image.size}")
plt.figure(figsize=(5, 4))
plt.imshow(real_image)
plt.title("Real image for CLIPSeg")
plt.axis("off")
plt.show()

## 2. 共享组件与超参数

为了让后面的 toy LSeg 和推理演示保持统一，本节集中定义：
- 张量维度与 patch 大小
- 文本特征构造方式
- 可视化函数
- 简单训练 / 评估工具

这样后面两部分都能复用同一套接口。

In [ ]:
# ── 共享超参数（CPU-friendly） ──
TOY_IMAGE_SIZE = 64
D_MODEL = 64
PATCH_SIZE = 8
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 128
DROPOUT = 0.0
LR = 3e-3
NUM_EPOCHS = 30
TOY_BATCH_SIZE = 16


def prompt_to_embedding(prompt, dim=D_MODEL):
    """用确定性随机向量模拟“冻结文本编码器”的输出。"""
    digest = hashlib.sha256(prompt.encode("utf-8")).digest()
    seed = int.from_bytes(digest[:8], "little")
    generator = torch.Generator().manual_seed(seed)
    vector = torch.randn(dim, generator=generator)
    return F.normalize(vector, dim=0)


def build_text_feature_bank(prompts, dim=D_MODEL, device=device):
    features = torch.stack([prompt_to_embedding(prompt, dim) for prompt in prompts], dim=0)
    return features.to(device)


toy_text_features = build_text_feature_bank(TOY_CLASS_NAMES)
print(f"toy_text_features shape: {tuple(toy_text_features.shape)}")  # (N, C)

In [ ]:
def pixel_accuracy(logits, target):
    pred = logits.argmax(dim=1)
    return (pred == target).float().mean().item()


def mean_iou(logits, target, num_classes):
    pred = logits.argmax(dim=1)
    scores = []
    for cls in range(num_classes):
        pred_mask = pred == cls
        target_mask = target == cls
        union = (pred_mask | target_mask).sum().item()
        if union == 0:
            continue
        inter = (pred_mask & target_mask).sum().item()
        scores.append(inter / union)
    return float(np.mean(scores)) if scores else float("nan")


def show_toy_predictions(images, targets, preds, max_items=3):
    max_items = min(max_items, images.shape[0])
    fig, axes = plt.subplots(max_items, 3, figsize=(9, 3 * max_items))
    axes = np.atleast_2d(axes)

    for row in range(max_items):
        axes[row, 0].imshow(np.transpose(images[row].cpu().numpy(), (1, 2, 0)))
        axes[row, 0].set_title("Input")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(targets[row].cpu().numpy(), cmap="viridis", vmin=0, vmax=3)
        axes[row, 1].set_title("Ground truth")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(preds[row].cpu().numpy(), cmap="viridis", vmin=0, vmax=3)
        axes[row, 2].set_title("Prediction")
        axes[row, 2].axis("off")

    plt.tight_layout()
    plt.show()

## 3. 架构解析：把 LSeg 的数据流拆开看

### 3.1 文本编码器（训练/推理全程冻结）

LSeg 的文本侧来自语言模型语义空间。设类别 prompt 数量为 $N$，通道维度为 $C$，则文本特征为：

$$
\mathbf{T} \in \mathbb{R}^{N \times C}
$$

其中第 $n$ 个类别的文本向量记为 $\mathbf{T}_n$。

**为什么冻结？**
- 分割数据远小于 CLIP 级别的图文数据规模；
- 希望保留已有的开放词汇语义空间；
- 训练时只让图像侧去适配文本语义，而不是把文本侧也一起改坏。

在这个 Notebook 里，我们不用真正训练文本编码器，而是用**确定性向量**模拟它的冻结输出，只保留“输入 prompt -> 输出固定文本特征”的接口。

In [ ]:
toy_text_features = build_text_feature_bank(TOY_CLASS_NAMES)
text_similarity = toy_text_features @ toy_text_features.T

print(f"Prompt bank: {TOY_CLASS_NAMES}")
print(f"Text feature bank shape: {tuple(toy_text_features.shape)}")  # (N, C)

plt.figure(figsize=(5, 4))
plt.imshow(text_similarity.cpu().numpy(), cmap="coolwarm", vmin=-1.0, vmax=1.0)
plt.xticks(range(len(TOY_CLASS_NAMES)), TOY_CLASS_NAMES, rotation=30, ha="right")
plt.yticks(range(len(TOY_CLASS_NAMES)), TOY_CLASS_NAMES)
plt.title("Prompt feature cosine similarity")
plt.colorbar()
plt.tight_layout()
plt.show()

### 3.2 图像编码器（DPT 风格密集特征）

LSeg 的图像侧不是输出单个全局向量，而是输出**密集空间特征图**：

$$
\mathbf{I} \in \mathbb{R}^{B \times C \times \tilde{H} \times \tilde{W}}
$$

这里的每个空间位置都对应一个局部视觉语义向量。为了便于 CPU 演示，下面实现一个 **DPT-style toy encoder**：
- `Patch Embedding`：把图像切成 patch；
- `Transformer Encoder`：建模 patch 之间的全局关系；
- `Decoder`：把 token 还原成稠密特征图。

### 3.3 像素级图文相似度

LSeg 的核心是把每个像素位置的视觉向量与所有文本类别向量比较：

$$
\mathbf{S}_{b,h,w,n} = \frac{\mathbf{I}_{b,:,h,w}^\top \mathbf{T}_n}{\|\mathbf{I}_{b,:,h,w}\|\,\|\mathbf{T}_n\|}
$$

得到的输出是：

$$
\mathbf{S} \in \mathbb{R}^{B \times N \times \tilde{H} \times \tilde{W}}
$$

### 3.4 Spatial Regularization Blocks

像素级相似度图往往比较粗糙，因此论文又加入 SRB 继续做空间平滑和局部细化。这里保留两种典型形式：

$$
\text{DepthwiseBlock}(x) = \sigma(\text{DWConv}(x))
$$

$$
\text{BottleneckBlock}(x) = \max(\sigma(\text{DWConv}(x)), x) + x
$$

它们的作用可以理解为：在不破坏类别维度的前提下，对空间预测做轻量 refinement。

In [ ]:
class SimpleDPTEncoder(nn.Module):
    def __init__(self, image_size=TOY_IMAGE_SIZE, patch_size=PATCH_SIZE, d_model=D_MODEL):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=NUM_HEADS,
            dim_feedforward=D_FF,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=NUM_LAYERS)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(d_model, d_model, kernel_size=2, stride=2),
            nn.GELU(),
            nn.ConvTranspose2d(d_model, d_model, kernel_size=2, stride=2),
            nn.GELU(),
            nn.Conv2d(d_model, d_model, kernel_size=1),
        )

    def forward(self, x):
        patches = self.patch_embed(x)                          # (B, 3, 64, 64) -> (B, d, 8, 8)
        h_patch, w_patch = patches.shape[-2:]
        tokens = patches.flatten(2).transpose(1, 2)           # (B, d, 8, 8) -> (B, 64, d)
        tokens = self.transformer(tokens)                     # (B, 64, d) -> (B, 64, d)
        features = tokens.transpose(1, 2).reshape(x.size(0), -1, h_patch, w_patch)
        dense = self.decoder(features)                        # (B, d, 8, 8) -> (B, d, 32, 32)
        return dense


def pixel_text_similarity(image_features, text_features):
    bsz, channels, height, width = image_features.shape
    image_tokens = image_features.flatten(2).transpose(1, 2)     # (B, C, H, W) -> (B, H*W, C)
    image_tokens = F.normalize(image_tokens, dim=-1)
    text_features = F.normalize(text_features, dim=-1)           # (N, C)
    scores = image_tokens @ text_features.T                      # (B, H*W, N)
    scores = scores.transpose(1, 2).reshape(bsz, -1, height, width)
    return scores                                                # (B, N, H, W)


class DepthwiseBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)                                     # (B, N, H, W) -> (B, N, H, W)


class BottleneckBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.depthwise = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels)
        self.act = nn.GELU()

    def forward(self, x):
        out = self.act(self.depthwise(x))                        # (B, N, H, W)
        out = torch.max(out, x)
        return out + x                                           # residual refinement


class ToyLSeg(nn.Module):
    def __init__(self, num_classes, image_size=TOY_IMAGE_SIZE, d_model=D_MODEL):
        super().__init__()
        self.image_encoder = SimpleDPTEncoder(image_size=image_size, patch_size=PATCH_SIZE, d_model=d_model)
        self.regularizer = nn.Sequential(
            DepthwiseBlock(num_classes),
            BottleneckBlock(num_classes),
        )
        self.upsample = nn.Upsample(size=(image_size, image_size), mode="bilinear", align_corners=False)

    def forward(self, images, text_features):
        dense_features = self.image_encoder(images)              # (B, d, 32, 32)
        similarity = pixel_text_similarity(dense_features, text_features)  # (B, N, 32, 32)
        refined = self.regularizer(similarity)                   # (B, N, 32, 32)
        logits = self.upsample(refined)                          # (B, N, 64, 64)
        return logits


toy_model = ToyLSeg(num_classes=len(TOY_CLASS_NAMES)).to(device)
with torch.inference_mode():
    sample_logits = toy_model(toy_images[:2].to(device), toy_text_features)

print(f"Sample logits shape: {tuple(sample_logits.shape)}")    # (B, N, H, W)
print(f"Expected target shape: {tuple(toy_masks[:2].shape)}")   # (B, H, W)

### 3.5 训练 vs 推理的区别

这是理解 LSeg 最关键的一段。

| 阶段 | 输入 | 输出 | 监督信号 | 目标 |
|---|---|---|---|---|
| Training | 图像 + **固定类别集合** 的文本特征 | `(B, N, H, W)` logits | 像素级标注 `y` | 让每个像素对正确类别文本向量响应更高 |
| Inference | 图像 + **任意 prompt** 文本特征 | `(B, N, H, W)` logits | 无 | 用 prompt 直接查询图像中是否存在该语义 |

训练和推理的共同点：都要做“密集视觉特征 × 文本特征”。

真正的区别在于：
- **训练** 有 `ground truth mask`，因此可以用交叉熵优化；
- **推理** 没有标签，只能把输出当作语义响应图来解释。

下面先把 training path 跑通，再看 inference path。

In [ ]:
def train_toy_lseg(model, text_features, epochs=NUM_EPOCHS, batch_size=TOY_BATCH_SIZE):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    history = {"loss": [], "pixel_acc": [], "miou": []}

    for epoch in range(epochs):
        model.train()
        images, masks = make_toy_batch(batch_size=batch_size, image_size=TOY_IMAGE_SIZE)
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images, text_features)                  # (B, N, H, W)
        loss = criterion(logits, masks)                        # target: (B, H, W)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history["loss"].append(loss.item())
        history["pixel_acc"].append(pixel_accuracy(logits.detach(), masks))
        history["miou"].append(mean_iou(logits.detach(), masks, num_classes=len(TOY_CLASS_NAMES)))

        if (epoch + 1) % 5 == 0:
            print(
                f"Epoch {epoch + 1:02d}/{epochs} | "
                f"loss={history['loss'][-1]:.4f} | "
                f"pixel_acc={history['pixel_acc'][-1]:.4f} | "
                f"miou={history['miou'][-1]:.4f}"
            )

    return history

In [ ]:
history = train_toy_lseg(toy_model, toy_text_features)

plt.figure(figsize=(12, 3.5))
plt.subplot(1, 3, 1)
plt.plot(history["loss"])
plt.title("Toy LSeg training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.subplot(1, 3, 2)
plt.plot(history["pixel_acc"])
plt.title("Toy LSeg pixel accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.subplot(1, 3, 3)
plt.plot(history["miou"])
plt.title("Toy LSeg mean IoU")
plt.xlabel("Epoch")
plt.ylabel("mIoU")

plt.tight_layout()
plt.show()

In [ ]:
toy_model.eval()
with torch.inference_mode():
    eval_images, eval_masks = make_toy_batch(batch_size=3, image_size=TOY_IMAGE_SIZE)
    eval_logits = toy_model(eval_images.to(device), toy_text_features)
    eval_preds = eval_logits.argmax(dim=1).cpu()

show_toy_predictions(eval_images, eval_masks, eval_preds)
print(f"Eval pixel accuracy: {pixel_accuracy(eval_logits.cpu(), eval_masks):.4f}")
print(f"Eval mIoU: {mean_iou(eval_logits.cpu(), eval_masks, len(TOY_CLASS_NAMES)):.4f}")

## 4. 推理应用：CLIPSeg Zero-Shot Segmentation

上面的 toy LSeg 展示了：
- 为什么训练需要像素标签；
- 为什么输出是 `(B, N, H, W)`；
- 文本特征如何变成逐像素分类头。

现在切换到真正可用的预训练模型 **CLIPSeg**。它和 LSeg 并不完全相同，但共享一个关键思想：
**把冻结的视觉-语言语义空间接到密集预测 decoder 上，让任意文本 prompt 直接驱动分割。**

根据官方 Transformers 文档，推荐调用链为：

```python
processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")
inputs = processor(text=texts, images=[image] * len(texts), padding=True, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits  # (num_prompts, 352, 352)
```

下面直接把它跑起来。

In [ ]:
clipseg_processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
clipseg_model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined").to(device)
clipseg_model.eval()

print("Loaded model: CIDAS/clipseg-rd64-refined")

In [ ]:
clipseg_inputs = clipseg_processor(
    text=REAL_PROMPTS,
    images=[real_image] * len(REAL_PROMPTS),
    padding=True,
    return_tensors="pt",
)
clipseg_inputs = {key: value.to(device) for key, value in clipseg_inputs.items()}

with torch.inference_mode():
    clipseg_outputs = clipseg_model(**clipseg_inputs)

clipseg_logits = clipseg_outputs.logits                      # (num_prompts, 352, 352)
clipseg_probs = torch.sigmoid(clipseg_logits).detach().cpu()

print(f"CLIPSeg logits shape: {tuple(clipseg_logits.shape)}")
print(f"Prompt count: {len(REAL_PROMPTS)}")

In [ ]:
fig, axes = plt.subplots(1, len(REAL_PROMPTS) + 1, figsize=(4 * (len(REAL_PROMPTS) + 1), 4))
axes[0].imshow(real_image)
axes[0].set_title("Original image")
axes[0].axis("off")

for idx, prompt in enumerate(REAL_PROMPTS):
    axes[idx + 1].imshow(clipseg_probs[idx].numpy(), cmap="magma")
    axes[idx + 1].set_title(prompt)
    axes[idx + 1].axis("off")

plt.suptitle("CLIPSeg prompt-conditioned masks", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
resized_masks = []
for idx in range(len(REAL_PROMPTS)):
    mask = Image.fromarray((clipseg_probs[idx].numpy() * 255).astype(np.uint8))
    mask = mask.resize(real_image.size, Image.BILINEAR)
    resized_masks.append(np.array(mask) / 255.0)

fig, axes = plt.subplots(1, len(REAL_PROMPTS), figsize=(4 * len(REAL_PROMPTS), 4))
if len(REAL_PROMPTS) == 1:
    axes = [axes]

for idx, prompt in enumerate(REAL_PROMPTS):
    axes[idx].imshow(real_image)
    axes[idx].imshow(resized_masks[idx], alpha=0.5, cmap="jet")
    axes[idx].set_title(prompt)
    axes[idx].axis("off")

plt.suptitle("CLIPSeg overlay results", y=1.02)
plt.tight_layout()
plt.show()

## 5. 结果对比与总结

### toy LSeg vs 预训练 CLIPSeg

| 维度 | toy LSeg | CLIPSeg |
|---|---|---|
| 目的 | 解释训练路径与张量流 | 直接运行 zero-shot segmentation |
| 文本特征 | Notebook 内部模拟冻结语义向量 | 真实预训练视觉-语言语义空间 |
| 监督 | 有，使用 toy mask + cross entropy | 无需当前任务标签，直接推理 |
| 输出 | 多类 logits `(B, N, H, W)` | 每个 prompt 一张二值响应图 `(N, 352, 352)` |
| 学习价值 | 明白 LSeg 为什么能训练 | 明白开放词汇分割怎么用 |

### 观察结论

- **LSeg 的本质**不是“把文本当标签名贴上去”，而是让每个像素进入语言语义空间。
- **训练阶段**的关键是像素级监督，把“哪个像素属于哪个语义”写进参数。
- **推理阶段**的关键是 prompt 查询：你换一组文本，输出语义响应图就会跟着变。
- **CLIPSeg 的 mask 较粗糙**，因为官方输出分辨率本身较低；做工程时通常还需要 resize、threshold、后处理。

## 面试拓展

### 高频面试题

**Q1：LSeg 为什么能做 zero-shot / open-vocabulary segmentation？**

- 因为类别不是固定写死在分类头里，而是来自文本编码器输出的语义向量。
- 图像编码器学到的是“language-aware dense feature”，每个像素都能和任意文本 prompt 做相似度。
- 所以推理时可以替换 prompt，而不一定要替换模型参数。

**Q2：LSeg 和 CLIP 最大的区别是什么？**

- CLIP 做的是 **image-level alignment**，输出通常是整张图一个全局向量。
- LSeg 做的是 **pixel-level alignment**，输出是每个空间位置的密集特征。
- 前者适合分类/检索，后者才能用于分割。

**Q3：为什么 LSeg 训练时常常冻结文本编码器？**

- 分割数据规模相对有限，难以稳定更新文本侧大模型。
- 冻结文本编码器可以保留已有语义空间，避免小数据把语言表示学坏。
- 工程上也能减少训练成本，把优化重点放在图像 encoder 和 decoder 上。

**Q4：为什么仅有像素级相似度还不够，还需要 Spatial Regularization Blocks？**

- 直接做相似度得到的响应图通常比较粗，局部噪声大。
- SRB 用轻量卷积在空间维度继续传播上下文，有助于边界与局部一致性。
- 它本质上是“相似度图后的 refinement 模块”。

**Q5：LSeg 的训练目标为什么通常是交叉熵，而不是 CLIP 式对比学习？**

- 因为分割任务需要**逐像素监督**，交叉熵可以直接约束每个像素的类别分布。
- 图像级对比学习不能直接告诉模型“哪一个像素错了”。
- 所以 LSeg 把视觉-语言表示引入分割任务，但训练目标仍然保留 dense prediction 的监督特征。

**Q6：CLIPSeg 和 LSeg 的关系怎么描述更准确？**

- 它们不是同一个模型。
- 但它们都属于“视觉-语言特征 + 稠密预测 decoder”的路线。
- 在教学里常把 CLIPSeg 作为 LSeg 的可运行近邻案例，因为它更容易直接推理演示。

**Q7：GroupViT 和 LSeg 的监督方式有什么本质不同？**

- LSeg 依赖像素级标注训练分割能力。
- GroupViT 强调从图文监督中学习 grouping，并在 zero-shot 语义分割上迁移。
- 因此 GroupViT 更偏“弱监督/文本监督 emergent segmentation”，而 LSeg 更偏“显式 supervised dense alignment”。

### 延伸阅读与对比

| 对比维度 | LSeg | CLIPSeg | GroupViT |
|---|---|---|---|
| 核心思想 | 像素特征与文本特征对齐 | 冻结 CLIP + decoder 做 prompt-conditioned mask | 从文本监督中学习视觉 grouping |
| 监督信号 | 像素级分割监督 | PhraseCut 等分割监督 | 图文对比监督，无像素标注 |
| 输出形式 | 多类语义分割 | 每个 prompt 一张二值 mask | 可迁移到 zero-shot segmentation |
| 工程上手难度 | 中 | 低 | 中 |
| 面试高频点 | 训练/推理差异、SRB、dense alignment | prompt inference、binary mask | weak supervision、group tokens |

### 进阶探索方向

- **开放词汇分割评测**：比较不同 prompt 写法对 mask 的影响，例如 `cat` vs `a cat` vs `the black cat`。
- **多 prompt 融合**：把多个二值 mask 组合成近似多类分割图。
- **后处理策略**：尝试 threshold、CRF、边界细化，观察低分辨率 mask 如何改善。
- **模型谱系梳理**：把 CLIP、LSeg、CLIPSeg、GroupViT、SAM 放到同一条发展线上总结。